# Strands Agents com AgentCore Memory (Memória de longo prazo) - Estratégias Integradas

## Introdução

Este tutorial demonstra como construir um **agente inteligente de suporte ao cliente** usando Strands Agents integrado com AgentCore Memory via hooks utilizando **MemoryManager** com **estratégias integradas** e **MemorySessionManager**. Vamos focar em memória de longo prazo para histórico de interações com clientes, lembrando detalhes de compras e fornecendo suporte personalizado com base em conversas anteriores e preferências do usuário.

**NOTA: Esta abordagem usa estratégias integradas (SemanticStrategy, UserPreferenceStrategy) que NÃO requerem uma IAM execution role.**

### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Conversacional de longo prazo                                                    |
| Tipo de agente      | Suporte ao Cliente                                                               |
| Framework de agentes| Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Extração de Memória Semântica e de Preferências do Usuário do AgentCore (Integrada), Hooks para armazenar e recuperar Memória |
| Complexidade do exemplo | Intermediário                                                                 |

Você aprenderá a:
- Configurar o AgentCore Memory com estratégias integradas de longo prazo usando MemoryManager
- Criar hooks de memória para armazenamento e recuperação automáticos com MemorySessionManager
- Construir um agente de suporte ao cliente com memória persistente
- Lidar com problemas de clientes usando contexto de interações anteriores
- Usar estratégias integradas sem necessidade de configuração de IAM role

### Contexto do Cenário
Neste exemplo, construiremos um **Caso de Uso de Suporte ao Cliente**. O agente lembrará o contexto do cliente, incluindo histórico de pedidos, preferências e problemas anteriores, permitindo um suporte mais personalizado e eficaz. As conversas com clientes são armazenadas automaticamente usando hooks de memória, garantindo que detalhes importantes nunca sejam perdidos. Ao empregar múltiplas estratégias de memória, como semântica e Preferência do Usuário — o agente pode capturar uma ampla gama de informações relevantes. Esta configuração permite que o agente resolva problemas com total consciência do histórico e preferências do cliente. Além disso, o agente está integrado com capacidades de busca na web, facilitando o fornecimento de informações atualizadas sobre produtos e orientações de solução de problemas conforme necessário.

## Arquitetura

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Pré-requisitos

Para executar este tutorial você precisará de:
- Python 3.10+
- Credenciais AWS com permissões do Amazon Bedrock AgentCore Memory
- SDK do Amazon Bedrock AgentCore com suporte ao MemoryManager

## 📊 Escolhendo entre Estratégias Integradas vs Personalizadas

O AgentCore Memory oferece duas abordagens para extração e consolidação de memória. Este notebook demonstra a abordagem de **Estratégias Integradas**.

### Estratégias Integradas (Este Notebook)

**Quando usar:**
- ✅ Configuração rápida e prototipação
- ✅ Necessidades padrão de extração de memória
- ✅ Sem necessidade de seleção personalizada de modelo
- ✅ Configuração simplificada de IAM (nenhuma execution role necessária)
- ✅ Cargas de trabalho em produção com modelos padrão do AgentCore

**Características principais:**
- Usa as classes `SemanticStrategy` e `UserPreferenceStrategy`
- O AgentCore Memory seleciona e gerencia modelos automaticamente
- Nenhuma IAM execution role necessária
- Configuração mais simples com menos parâmetros
- Ideal para a maioria dos casos de uso

**Exemplo:**
```python
strategies = [
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies
    # No memory_execution_role_arn needed!
)
```

### Substituição por Estratégia Personalizada (Veja `customer-support-override-strategy.ipynb`)

**Quando usar:**
- ✅ Necessidade de especificar modelos Bedrock personalizados para extração/consolidação
- ✅ Controle refinado sobre o comportamento do modelo
- ✅ Prompts personalizados para extração e consolidação
- ✅ Casos de uso avançados que requerem capacidades específicas do modelo
- ✅ Requisitos de conformidade para versões específicas de modelo

**Características principais:**
- Usa as classes `CustomSemanticStrategy` e `CustomUserPreferenceStrategy`
- Requer IAM execution role para invocação do modelo
- Permite especificação de `ExtractionConfig` e `ConsolidationConfig`
- Configuração mais complexa, porém com maior controle
- Útil para requisitos especializados

**Exemplo:**
```python
strategies = [
    CustomSemanticStrategy(
        name="CustomerSupportSemantic",
        extraction_config=ExtractionConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Extract factual information..."
        ),
        consolidation_config=ConsolidationConfig(
            model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
            append_to_prompt="Consolidate semantic insights..."
        ),
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

memory = memory_manager.get_or_create_memory(
    name="CustomerSupportMemory",
    strategies=strategies,
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN  # Required!
)
```

### Tabela Comparativa Rápida

| Funcionalidade | Estratégias Integradas | Substituição por Estratégia Personalizada |
|---------|---------------------|-------------------------|
| **Complexidade de Configuração** | Simples | Avançada |
| **IAM Role Necessária** | ❌ Não | ✅ Sim |
| **Seleção de Modelo** | Automática (gerenciada pelo AgentCore) | Manual (você especifica) |
| **Prompts Personalizados** | ❌ Não | ✅ Sim |
| **Configuração** | Mínima | Detalhada |
| **Caso de Uso** | Extração de memória padrão | Requisitos de modelo personalizados |
| **Recomendado Para** | Maioria das aplicações | Necessidades especializadas |

---

**💡 Recomendação:** Comece com estratégias integradas (este notebook) para a maioria dos casos de uso. Use a substituição por estratégia personalizada apenas se você tiver requisitos específicos de modelo ou precisar de controle refinado sobre o comportamento de extração/consolidação.

## Passo 1: Instalar Dependências e Configuração
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para que este notebook funcione.

In [19]:
!pip install -qr requirements.txt

In [20]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Import required modules for Strands Agent
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS

# Import memory management modules
from bedrock_agentcore_starter_toolkit.operations.memory.manager import Memory, MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy, UserPreferenceStrategy
)
from bedrock_agentcore.memory.constants import (
    BlobMessage, ConversationalMessage, MessageRole, RetrievalConfig
)
from bedrock_agentcore.memory.models import (
    StringValue, EventMetadataFilter, LeftExpression, RightExpression, OperatorType, MemoryRecord
)
from bedrock_agentcore.memory.session import Actor, MemorySession, MemorySessionManager

# Define message role constants
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

logger.info("✅ All imports loaded successfully")


2026-03-24 19:04:54,510 - INFO - ✅ All imports loaded successfully


In [21]:
# Configuration - Replace with the correct values
REGION = "us-east-1"  
CUSTOMER_ID = "customer_001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"✅ Configuration loaded")
logger.info(f"   Region: {REGION}")
logger.info(f"   Customer ID: {CUSTOMER_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

2026-03-24 19:04:54,514 - INFO - ✅ Configuration loaded
2026-03-24 19:04:54,514 - INFO -    Region: us-east-1
2026-03-24 19:04:54,515 - INFO -    Customer ID: customer_001
2026-03-24 19:04:54,515 - INFO -    Session ID: support_20260324190454


## Passo 2: Criar Recurso de Memória para Suporte ao Cliente

Para suporte ao cliente, usaremos múltiplas estratégias integradas de memória:
- **UserPreferenceStrategy**: Captura preferências e comportamento do cliente (integrada)
- **SemanticStrategy**: Armazena fatos sobre pedidos e informações de produtos (integrada)

**IMPORTANTE**: Estratégias integradas NÃO requerem uma IAM execution role. O AgentCore Memory usa modelos padrão para extração e consolidação.

In [22]:
# Initialize Memory Manager 
memory_manager = MemoryManager(region_name=REGION)
memory_name = "CustomerSupportLongTermMemory"

# Test basic memory manager initialization and connection
logger.info(f"✅ MemoryManager initialized for region: {REGION}")
logger.info(f"Memory manager type: {type(memory_manager)}")

# Define memory strategies using built-in strategy classes
strategies = [
    UserPreferenceStrategy(
        name="CustomerPreferences",
        description="Captures customer preferences and behavior",
        namespaces=["support/customer/{actorId}/preferences/"]
    ),
    SemanticStrategy(
        name="CustomerSupportSemantic",
        description="Stores facts from conversations",
        namespaces=["support/customer/{actorId}/semantic/"]
    )
]

# Validate strategy configuration
logger.info(f"✅ Configured {len(strategies)} built-in memory strategies:")
for i, strategy in enumerate(strategies, 1):
    logger.info(f"  {i}. {strategy.name} ({type(strategy).__name__})")
    logger.info(f"     Description: {strategy.description}")
    logger.info(f"     Namespaces: {strategy.namespaces}")
    logger.info(f"     Uses default AgentCore models (no IAM role required)")

# Create memory resource using MemoryManager with built-in strategies
logger.info(f"Creating memory '{memory_name}' with {len(strategies)} built-in strategies...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,         # Pass built-in strategy objects
        description="Memory for customer support agent with built-in strategies",
        event_expiry_days=90,          # Memories expire after 90 days
        # NOTE: No memory_execution_role_arn needed for built-in strategies
    )
    memory_id = memory.id
    logger.info(f"✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
    logger.info(f"   Using built-in strategies (no IAM role required)")
    
except Exception as e:
    # Handle any errors during memory creation with enhanced error reporting
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")
    
    # Re-raise the original exception
    raise

2026-03-24 19:04:54,529 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
✅ MemoryManager initialized for region: us-east-1
2026-03-24 19:04:54,646 - INFO - ✅ MemoryManager initialized for region: us-east-1
2026-03-24 19:04:54,649 - INFO - ✅ MemoryManager initialized for region: us-east-1
2026-03-24 19:04:54,650 - INFO - Memory manager type: <class 'bedrock_agentcore_starter_toolkit.operations.memory.manager.MemoryManager'>
2026-03-24 19:04:54,650 - INFO - ✅ Configured 2 built-in memory strategies:
2026-03-24 19:04:54,651 - INFO -   1. CustomerPreferences (UserPreferenceStrategy)
2026-03-24 19:04:54,651 - INFO -      Description: Captures customer preferences and behavior
2026-03-24 19:04:54,651 - INFO -      Namespaces: ['support/customer/{actorId}/preferences/']
2026-03-24 19:04:54,651 - INFO -      Uses default AgentCore models (no IAM role required)
2026-03-24 19:04:54,651 - INFO -   2. CustomerSupportSemantic (SemanticStrategy)
2026-03-24 19:04:54,652 - INF

In [23]:
# Verify memory manager connection and display existing memories
try:
    existing_memories = memory_manager.list_memories()
    logger.info(f"✅ Memory manager connection successful. Found {len(existing_memories)} existing memories")
    for mem in existing_memories:
        logger.info(f"   - {mem.name} ({mem.id}) - Status: {mem.status}")
except Exception as e:
    logger.error(f"❌ Memory manager test failed: {e}")
    raise

2026-03-24 19:07:50,749 - INFO - ✅ Memory manager connection successful. Found 2 existing memories
2026-03-24 19:07:50,750 - INFO -    - None (CustomerSupportLongTermMemory-KV3MD7HT8f) - Status: ACTIVE
2026-03-24 19:07:50,750 - INFO -    - None (SelfManageMemory1774388397-iupKhfBSES) - Status: ACTIVE


Vamos confirmar se nossa memória contém as estratégias que atribuímos

In [24]:
# Display memory information using MemoryManager
print(f"Memory ID: {memory.id}")
print(f"Memory Name: {memory.name}")
print(f"Memory Description: {memory.description}")
print(f"Memory Status: {memory.status}")
print(f"Number of strategies: {len(strategies)}")
for i, strategy in enumerate(strategies, 1):
    print(f"  {i}. {strategy.name}: {strategy.description}")

Memory ID: CustomerSupportLongTermMemory-KV3MD7HT8f
Memory Name: CustomerSupportLongTermMemory
Memory Description: Memory for customer support agent with built-in strategies
Memory Status: CREATING
Number of strategies: 2
  1. CustomerPreferences: Captures customer preferences and behavior
  2. CustomerSupportSemantic: Stores facts from conversations


## Passo 3: Criar Ferramentas do Agente

In [25]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for product information, troubleshooting guides, or support articles.
    
    Args:
        query: Search query for product info or troubleshooting
        max_results: Maximum number of results to return
    
    Returns:
        Search results with titles and snippets
    """
    try:
        results = DDGS().text(query, region="us-en", max_results=max_results)
        if not results:
            return "No search results found."
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('body', 'No description')}")
        
        return "\n".join(formatted_results)
    except RatelimitException:
        return "Rate limit reached: Please try again after a short delay."
    except DDGSException as d:
        return f"Search Error: {d}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Web search tool ready")

@tool
def check_order_status(order_number: str) -> str:
    """Check the status of a customer order.
    
    Args:
        order_number: The order number to check
    
    Returns:
        Order status information
    """
    # Simulate order lookup
    mock_orders = {
        "123456": "iPhone 15 Pro - Delivered on June 5, 2025",
        "654321": "Sennheiser Headphones - Delivered on June 25, 2025, 1-year warranty active",
        "789012": "Samsung Galaxy S23 - In transit, expected delivery on July 1, 2025",
    }
    
    return mock_orders.get(order_number, f"Order {order_number} not found. Please verify the order number.")

logger.info("✅ Check Order Status tool ready")

2026-03-24 19:07:50,766 - INFO - ✅ Web search tool ready
2026-03-24 19:07:50,768 - INFO - ✅ Check Order Status tool ready


## Passo 4: Inicializar o Gerenciador de Sessão

**NOVO: Esta seção apresenta o MemorySessionManager para operações de Memória baseadas em sessão.**

In [26]:
# Initialize the session memory manager
session_manager: MemorySessionManager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# Create a memory session for the specific customer
customer_session: MemorySession = session_manager.create_memory_session(
    actor_id=CUSTOMER_ID, 
    session_id=SESSION_ID
)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"✅ Customer session created for actor: {CUSTOMER_ID}")
logger.info(f"   Session type: {type(customer_session)}")
logger.info(f"   Actor object: {customer_session.get_actor()}")

2026-03-24 19:07:50,783 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-24 19:07:50,901 - INFO - 💬 Creating new conversation for actor 'customer_001' in session 'support_20260324190454'...
2026-03-24 19:07:50,901 - INFO - ✅ Session manager initialized for memory: CustomerSupportLongTermMemory-KV3MD7HT8f
2026-03-24 19:07:50,902 - INFO - ✅ Customer session created for actor: customer_001
2026-03-24 19:07:50,902 - INFO -    Session type: <class 'bedrock_agentcore.memory.session.MemorySession'>
2026-03-24 19:07:50,902 - INFO -    Actor object: {'actorId': 'customer_001'}


## Passo 5: Criar Hook Provider de Memória para Suporte ao Cliente
Hooks são funções especiais que são executadas em pontos específicos do ciclo de vida de execução de um agente. Nosso hook provider personalizado gerenciará automaticamente o contexto de suporte ao cliente:
- **Salvando interações de suporte** após cada resposta usando métodos baseados em sessão
- **Recuperando e injetando contexto relevante** de pedidos e preferências anteriores ao processar novas consultas.


In [27]:
class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent - ENHANCED with MemorySession"""
    
    def __init__(self, customer_session: MemorySession):
        # Accept MemorySession directly 
        self.customer_session = customer_session
        
        # Define retrieval configuration for different memory types
        self.retrieval_config = {
            "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
            "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=5, relevance_score=0.2)
        }
    
    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query using MemorySession"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # Use MemorySession for context retrieval
                relevant_memories = []
                
                # Search across different memory namespaces using MemorySession
                for namespace_template, config in self.retrieval_config.items():
                    # Resolve namespace template with actual actor ID from session
                    resolved_namespace = namespace_template.format(
                        actorId=self.customer_session._actor_id
                    )
                    
                    # Use MemorySession API (no need to pass actor_id/session_id)
                    memories = self.customer_session.search_long_term_memories(
                        query=user_query,
                        namespace_prefix=resolved_namespace,
                        top_k=config.top_k
                    )
                    
                    # Filter by relevance score
                    filtered_memories = [
                        memory for memory in memories
                        if memory.get("score", 0) >= config.relevance_score
                    ]
                    
                    relevant_memories.extend(filtered_memories)
                    logger.info(f"Found {len(filtered_memories)} relevant memories in {resolved_namespace} (filtered from {len(memories)} total)")
                
                # Inject context into agent's system prompt if memories found
                if relevant_memories:
                    context_text = self._format_context(relevant_memories)
                    original_prompt = event.agent.system_prompt
                    enhanced_prompt = f"{original_prompt}\n\nCustomer Context:\n{context_text}"
                    event.agent.system_prompt = enhanced_prompt
                    logger.info(f"✅ Injected {len(relevant_memories)} memories into agent context")
                    
            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")
    
    def _format_context(self, memories: List[MemoryRecord]) -> str:
        """Format retrieved memories for agent context"""
        context_lines = []
        for i, memory in enumerate(memories[:5], 1):  # Limit to top 5
            content = memory.get('content', {}).get('text', 'No content available')
            score = memory.get('score', 0)
            context_lines.append(f"{i}. (Score: {score:.2f}) {content[:200]}...")
        
        return "\n".join(context_lines)
    
    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save support interaction using MemorySession (cleaner API)"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                if customer_query and agent_response:
                    # Use MemorySession (no need to pass actor_id/session_id)
                    interaction_messages = [
                        ConversationalMessage(customer_query, USER),
                        ConversationalMessage(agent_response, ASSISTANT)
                    ]
                    
                    result = self.customer_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved interaction using MemorySession - Event ID: {result['eventId']}")
                    
        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)  # Re-added!
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("✅ Customer support memory hooks registered with MemorySession")
print('Executed!')

Executed!


### Passo 6: Criar Agente de Suporte ao Cliente

In [28]:
# Create memory hooks using MemorySession
support_hooks = CustomerSupportMemoryHooks(customer_session)

# Create customer support agent
support_agent = Agent(
    hooks=[support_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[web_search, check_order_status],
    system_prompt="""You are a helpful customer support agent with access to customer history and order information. 
    
    Your role:
    - Help customers with their orders, returns, and product issues
    - Use customer context to provide personalized support
    - Search for product information when needed
    - Be empathetic and solution-focused
    - Reference previous orders and preferences when relevant
    
    Always be professional, helpful, and aim to resolve customer issues efficiently."""
)

print("✅ Customer support agent created with MemorySession integration")

2026-03-24 19:07:50,921 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-24 19:07:51,019 - INFO - ✅ Customer support memory hooks registered with MemorySession


✅ Customer support agent created with MemorySession integration


### Passo 7: Alimentar Histórico do Cliente

Vamos adicionar algumas interações anteriores do cliente para demonstrar a funcionalidade de memória.

**NOTA: Esta seção usa o formato ConversationalMessage e armazenamento baseado em sessão.**

In [29]:
# Seed with previous customer interactions using MemorySession
previous_interactions = [
    ConversationalMessage("I bought a new iPhone 15 Pro on June 1st, 2025. Order number is 123456.", USER),
    ConversationalMessage("Thank you for your purchase! I can see your iPhone 15 Pro order #123456 was delivered successfully. How can I help you today?", ASSISTANT),
    ConversationalMessage("I also ordered Sennheiser headphones on June 20th. Order number 654321. They came with 1-year warranty.", USER),
    ConversationalMessage("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty. Both your iPhone and headphones should work great together.", ASSISTANT),
    ConversationalMessage("I'm looking for a good laptop. I prefer ThinkPad models.", USER),
    ConversationalMessage("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", ASSISTANT)
]

# Save using MemorySession 
try:
    event_response = customer_session.add_turns(previous_interactions)
    logger.info(f"✅ Seeded customer history using MemorySession")
    logger.info(f"   Event ID: {event_response['eventId']}")
except Exception as e:
    logger.error(f"⚠️ Error seeding history: {e}")


2026-03-24 19:07:51,028 - INFO -   -> Storing 6 messages in short-term memory...
2026-03-24 19:07:51,587 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:07:51,588 - INFO - ✅ Seeded customer history using MemorySession
2026-03-24 19:07:51,588 - INFO -    Event ID: 0000001774390071029#dfd7acc5


#### O agente está pronto.

### Vamos testar Cenários de Suporte ao Cliente

In [30]:
# Test 1: Customer reports iPhone issue 
logger.info("🧪 Running Test 1: iPhone performance issue")
test_query_1 = "My iPhone is running very slow and gets hot when charging. Can you help?"
logger.info(f"Query: {test_query_1}")

response1 = support_agent(test_query_1)
logger.info(f"✅ Test 1 completed successfully")
print(f"\n📱 iPhone Issue Support Response:\n{response1}\n")

2026-03-24 19:07:51,595 - INFO - 🧪 Running Test 1: iPhone performance issue
2026-03-24 19:07:51,596 - INFO - Query: My iPhone is running very slow and gets hot when charging. Can you help?
2026-03-24 19:07:51,597 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/preferences/' with query: 'My iPhone is running very slow and gets hot when charging. Can you help?'...
2026-03-24 19:07:52,013 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:07:52,014 - INFO - Found 0 relevant memories in support/customer/customer_001/preferences/ (filtered from 0 total)
2026-03-24 19:07:52,014 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/semantic/' with query: 'My iPhone is running very slow and gets hot when charging. Can you help?'...
2026-03-24 19:07:52,393 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:07:52,394 - INFO - Found 0 relevant memories in support/customer/customer_001/semantic/ (filter

I'd be happy to help you troubleshoot your iPhone issues! Let me search for some helpful information about slow performance and overheating while charging.
Tool #1: web_search

Tool #2: web_search


2026-03-24 19:07:54,536 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=iPhone%20slow%20performance%20solutions%20fix 200
2026-03-24 19:07:54,561 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=iPhone%20running%20slow%20overheating%20when%20charging%20troubleshooting 200
2026-03-24 19:07:55,168 - INFO - response: https://grokipedia.com/api/typeahead?query=iPhone+slow+performance+solutions+fix&limit=1 200
2026-03-24 19:07:55,168 - INFO - response: https://grokipedia.com/api/typeahead?query=iPhone+running+slow+overheating+when+charging+troubleshooting&limit=1 200
2026-03-24 19:07:56,013 - INFO - response: https://www.mojeek.com/search?q=iPhone+running+slow+overheating+when+charging+troubleshooting 403
2026-03-24 19:07:56,013 - INFO - response: https://www.mojeek.com/search?q=iPhone+slow+performance+solutions+fix 403
2026-03-24 19:07:56,770 - INFO - response: https://search.brave.com/

Great! I found some helpful information. Here are some solutions to try for your slow iPhone and overheating issues:

## For Overheating While Charging:

1. **Remove the case** - Cases can trap heat. Try charging without it temporarily
2. **Stop using resource-intensive apps** - Close apps running in the background, especially while charging
3. **Avoid direct sunlight** - Charge in a cool location, not in direct sun or hot environments
4. **Use the original charger** - Ensure you're using an Apple-certified charger, as incompatible chargers can cause overheating

## For Overall Slow Performance:

1. **Close background apps** - Swipe up from the bottom of your screen and close apps running in the background
2. **Check your storage** - Clear out unnecessary files and apps, especially if you're running low on storage ("Other" storage can accumulate)
3. **Check your Wi-Fi connection** - A faulty or misconfigured network connection can slow performance
4. **Restart your iPhone** - Sometimes

2026-03-24 19:08:00,873 - INFO -   -> Storing 2 messages in short-term memory...


?

2026-03-24 19:08:01,172 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:01,173 - INFO - ✅ Saved interaction using MemorySession - Event ID: 0000001774390080874#4bdbf81c
2026-03-24 19:08:01,176 - INFO - ✅ Test 1 completed successfully



📱 iPhone Issue Support Response:
Great! I found some helpful information. Here are some solutions to try for your slow iPhone and overheating issues:

## For Overheating While Charging:

1. **Remove the case** - Cases can trap heat. Try charging without it temporarily
2. **Stop using resource-intensive apps** - Close apps running in the background, especially while charging
3. **Avoid direct sunlight** - Charge in a cool location, not in direct sun or hot environments
4. **Use the original charger** - Ensure you're using an Apple-certified charger, as incompatible chargers can cause overheating

## For Overall Slow Performance:

1. **Close background apps** - Swipe up from the bottom of your screen and close apps running in the background
2. **Check your storage** - Clear out unnecessary files and apps, especially if you're running low on storage ("Other" storage can accumulate)
3. **Check your Wi-Fi connection** - A faulty or misconfigured network connection can slow performance
4. *

In [31]:
# Test 2: Bluetooth connectivity issue 
logger.info("🧪 Running Test 2: Bluetooth connectivity issue")
test_query_2 = "My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?"
logger.info(f"Query: {test_query_2}")

response2 = support_agent(test_query_2)
logger.info(f"✅ Test 2 completed successfully")
print(f"\n🎧 Bluetooth Issue Support Response:\n{response2}\n")

2026-03-24 19:08:01,181 - INFO - 🧪 Running Test 2: Bluetooth connectivity issue
2026-03-24 19:08:01,182 - INFO - Query: My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?
2026-03-24 19:08:01,183 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/preferences/' with query: 'My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?'...
2026-03-24 19:08:01,461 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:01,462 - INFO - Found 0 relevant memories in support/customer/customer_001/preferences/ (filtered from 0 total)
2026-03-24 19:08:01,462 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/semantic/' with query: 'My iPhone won't connect to my Sennheiser headphones via Bluetooth. How do I fix this?'...
2026-03-24 19:08:01,767 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:01,768 - INFO - Found 0 relevant memories in supp

Let me search for some troubleshooting steps to help you get your Sennheiser headphones connected to your iPhone via Bluetooth.
Tool #3: web_search

Tool #4: web_search


2026-03-24 19:08:03,883 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=iPhone%20won%27t%20connect%20Sennheiser%20headphones%20Bluetooth%20troubleshooting 200
2026-03-24 19:08:03,902 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=fix%20iPhone%20Bluetooth%20connection%20issues%20headphones 200
2026-03-24 19:08:04,109 - INFO - response: https://grokipedia.com/api/typeahead?query=iPhone+won%27t+connect+Sennheiser+headphones+Bluetooth+troubleshooting&limit=1 200
2026-03-24 19:08:04,181 - INFO - response: https://grokipedia.com/api/typeahead?query=fix+iPhone+Bluetooth+connection+issues+headphones&limit=1 200
2026-03-24 19:08:04,556 - INFO - response: https://www.google.com/search?q=iPhone+won%27t+connect+Sennheiser+headphones+Bluetooth+troubleshooting&filter=1&start=0&hl=en-US&lr=lang_en&cr=countryUS 403
2026-03-24 19:08:04,881 - INFO - response: https://www.mojeek.com/search?q=fix+iPh

Great! Here are some troubleshooting steps to get your Sennheiser headphones connected to your iPhone:

## Basic Steps:

1. **Ensure Bluetooth is enabled on your iPhone**
   - Go to Settings > Bluetooth
   - Toggle Bluetooth ON if it's off

2. **Put your Sennheiser headphones in pairing mode**
   - Check your headphone's manual for the specific pairing button (usually a dedicated button or holding a button for several seconds)
   - Look for a LED indicator that shows pairing mode (usually blinking)

3. **Make sure headphones are charged and powered on**
   - Verify they have sufficient battery
   - Check if batteries need to be replaced (if battery-operated)

## If That Doesn't Work:

4. **Forget the device and re-pair**
   - Go to Settings > Bluetooth
   - Find your Sennheiser headphones in the list
   - Tap the "i" icon and select "Forget This Device"
   - Put headphones back in pairing mode and try connecting again

5. **Restart both devices**
   - Turn off your Sennheiser headphone

2026-03-24 19:08:10,326 - INFO -   -> Storing 2 messages in short-term memory...


 you more targeted help!

2026-03-24 19:08:10,650 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:10,651 - INFO - ✅ Saved interaction using MemorySession - Event ID: 0000001774390090327#a9bc198c
2026-03-24 19:08:10,654 - INFO - ✅ Test 2 completed successfully



🎧 Bluetooth Issue Support Response:
Great! Here are some troubleshooting steps to get your Sennheiser headphones connected to your iPhone:

## Basic Steps:

1. **Ensure Bluetooth is enabled on your iPhone**
   - Go to Settings > Bluetooth
   - Toggle Bluetooth ON if it's off

2. **Put your Sennheiser headphones in pairing mode**
   - Check your headphone's manual for the specific pairing button (usually a dedicated button or holding a button for several seconds)
   - Look for a LED indicator that shows pairing mode (usually blinking)

3. **Make sure headphones are charged and powered on**
   - Verify they have sufficient battery
   - Check if batteries need to be replaced (if battery-operated)

## If That Doesn't Work:

4. **Forget the device and re-pair**
   - Go to Settings > Bluetooth
   - Find your Sennheiser headphones in the list
   - Tap the "i" icon and select "Forget This Device"
   - Put headphones back in pairing mode and try connecting again

5. **Restart both devices**
  

In [32]:
# Test 3: Check order status 
logger.info("🧪 Running Test 3: Order status check")
test_query_3 = "Can you check the status of my recent orders?"
logger.info(f"Query: {test_query_3}")

response3 = support_agent(test_query_3)
logger.info(f"✅ Test 3 completed successfully")
print(f"\n📦 Order Status Support Response:\n{response3}\n")

2026-03-24 19:08:10,661 - INFO - 🧪 Running Test 3: Order status check
2026-03-24 19:08:10,661 - INFO - Query: Can you check the status of my recent orders?
2026-03-24 19:08:10,662 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/preferences/' with query: 'Can you check the status of my recent orders?'...
2026-03-24 19:08:11,039 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:11,039 - INFO - Found 0 relevant memories in support/customer/customer_001/preferences/ (filtered from 0 total)
2026-03-24 19:08:11,039 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/semantic/' with query: 'Can you check the status of my recent orders?'...
2026-03-24 19:08:11,497 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:11,497 - INFO - Found 0 relevant memories in support/customer/customer_001/semantic/ (filtered from 0 total)


I'd be happy to help you check the status of your recent orders! However, I need your order number(s) to look that up.

Could you please provide me with:
- **Your order number(s)** - This is typically found in your confirmation email or on your account

Once you give me the order number(s), I'll be able to check the status for

2026-03-24 19:08:13,200 - INFO -   -> Storing 2 messages in short-term memory...


 you right away!

2026-03-24 19:08:13,492 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:13,492 - INFO - ✅ Saved interaction using MemorySession - Event ID: 0000001774390093201#c0640ebe
2026-03-24 19:08:13,494 - INFO - ✅ Test 3 completed successfully



📦 Order Status Support Response:
I'd be happy to help you check the status of your recent orders! However, I need your order number(s) to look that up.

Could you please provide me with:
- **Your order number(s)** - This is typically found in your confirmation email or on your account

Once you give me the order number(s), I'll be able to check the status for you right away!




In [33]:
# Test 4: Product recommendation based on preferences 
logger.info("🧪 Running Test 4: Product recommendation")
test_query_4 = "I'm still interested in buying a laptop. What ThinkPad models do you recommend?"
logger.info(f"Query: {test_query_4}")

response4 = support_agent(test_query_4)
logger.info(f"✅ Test 4 completed successfully")
print(f"\n💻 Product Recommendation Support Response:\n{response4}\n")

logger.info("🎉 All customer support scenario tests completed!")

2026-03-24 19:08:13,500 - INFO - 🧪 Running Test 4: Product recommendation
2026-03-24 19:08:13,501 - INFO - Query: I'm still interested in buying a laptop. What ThinkPad models do you recommend?
2026-03-24 19:08:13,501 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/preferences/' with query: 'I'm still interested in buying a laptop. What ThinkPad models do you recommend?'...
2026-03-24 19:08:13,824 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:13,825 - INFO - Found 0 relevant memories in support/customer/customer_001/preferences/ (filtered from 0 total)
2026-03-24 19:08:13,825 - INFO -   -> Querying long-term memory in namespace 'support/customer/customer_001/semantic/' with query: 'I'm still interested in buying a laptop. What ThinkPad models do you recommend?'...
2026-03-24 19:08:14,183 - INFO -      ✅ Found 0 relevant long-term records.
2026-03-24 19:08:14,183 - INFO - Found 0 relevant memories in support/customer/customer_00

I appreciate your interest in purchasing a ThinkPad laptop! However, I should let you know that I'm specifically here to help with customer support for orders, returns, and product issues related to items you've already purchased or are having problems with.

For product recommendations and purchasing advice on new laptops like ThinkPad models, I'd recommend:

1. **Visiting the official Lenovo website** - They have detailed specifications and comparisons of different ThinkPad models
2. **Contacting Lenovo sales** - Their sales team can help you find the right model for your needs and budget
3. **Reading reviews** - Tech review sites often have helpful comparisons of current ThinkPad models

**However, if you:**
- Have an existing order you'd like me to check on
- Need help with a ThinkPad or other product you've already purchased
- Have questions about a return or exchange

...I'm here to help with those! Just let me know your order number or details about the issue you're experiencing

2026-03-24 19:08:17,125 - INFO -   -> Storing 2 messages in short-term memory...


 an existing purchase?

2026-03-24 19:08:17,415 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:17,416 - INFO - ✅ Saved interaction using MemorySession - Event ID: 0000001774390097126#316543f4
2026-03-24 19:08:17,418 - INFO - ✅ Test 4 completed successfully
2026-03-24 19:08:17,419 - INFO - 🎉 All customer support scenario tests completed!



💻 Product Recommendation Support Response:
I appreciate your interest in purchasing a ThinkPad laptop! However, I should let you know that I'm specifically here to help with customer support for orders, returns, and product issues related to items you've already purchased or are having problems with.

For product recommendations and purchasing advice on new laptops like ThinkPad models, I'd recommend:

1. **Visiting the official Lenovo website** - They have detailed specifications and comparisons of different ThinkPad models
2. **Contacting Lenovo sales** - Their sales team can help you find the right model for your needs and budget
3. **Reading reviews** - Tech review sites often have helpful comparisons of current ThinkPad models

**However, if you:**
- Have an existing order you'd like me to check on
- Need help with a ThinkPad or other product you've already purchased
- Have questions about a return or exchange

...I'm here to help with those! Just let me know your order number or

## Funcionalidades Avançadas: Ramificação e Metadados

### Ramificação de Conversas com SessionManager

Explore cenários alternativos de suporte usando ramificação:

In [34]:
# Get the last event ID from our conversation
events = customer_session.list_events()
if events:
    last_event_id = events[-1].eventId
    
    # Fork conversation to explore premium support path
    branch_event = customer_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="premium-support",
        messages=[
            ConversationalMessage(
                "I'd like to upgrade to premium support for faster resolution.",
                USER
            ),
            ConversationalMessage(
                "Excellent choice! With premium support, you'll get 24/7 priority assistance, dedicated account manager, and same-day resolution guarantee. Let me process your upgrade.",
                ASSISTANT
            )
        ]
    )
    
    logger.info(f"✅ Created premium support branch from event {last_event_id}")
    
    # List all branches
    branches = customer_session.list_branches()
    print(f"\n🌳 Support session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")
else:
    print("No events found to branch from")

2026-03-24 19:08:17,622 - INFO - Retrieved total of 5 events
2026-03-24 19:08:17,623 - INFO -   -> Storing 2 messages in short-term memory...
2026-03-24 19:08:17,919 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:17,920 - INFO - Created branch 'premium-support' from event 0000001774390071029#dfd7acc5
2026-03-24 19:08:17,920 - INFO - ✅ Created premium support branch from event 0000001774390071029#dfd7acc5
2026-03-24 19:08:18,140 - INFO - Found 2 branches in session support_20260324190454



🌳 Support session has 2 branch(es):
   - premium-support: None events
   - main: None events


### Metadados para Rastreamento Avançado de Suporte

Use metadados para rastrear métricas abrangentes de suporte:

In [35]:
from bedrock_agentcore.memory.models import StringValue

# Add a support interaction with comprehensive metadata
metadata_event = customer_session.add_turns(
    messages=[
        ConversationalMessage(
            "The ThinkPad X1 Carbon you recommended is perfect! I'll order it now.",
            USER
        ),
        ConversationalMessage(
            "Fantastic! The ThinkPad X1 Carbon is an excellent choice for your needs. I'll help you complete the order with your preferred configuration.",
            ASSISTANT
        )
    ],
    metadata={
        "interaction_type": StringValue.build("product_recommendation"),
        "outcome": StringValue.build("purchase_intent"),
        "product_category": StringValue.build("laptops"),
        "product_brand": StringValue.build("lenovo"),
        "customer_sentiment": StringValue.build("positive"),
        "support_tier": StringValue.build("standard"),
        "session_duration_minutes": StringValue.build("15")
    }
)

logger.info(f"✅ Added support event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Support interaction tagged with:")
print("   - Interaction Type: product_recommendation")
print("   - Outcome: purchase_intent")
print("   - Product Category: laptops")
print("   - Customer Sentiment: positive")

2026-03-24 19:08:18,149 - INFO -   -> Storing 2 messages in short-term memory...
2026-03-24 19:08:18,435 - INFO -      ✅ Turn stored successfully with Event ID: None
2026-03-24 19:08:18,436 - INFO - ✅ Added support event with metadata - Event ID: 0000001774390098150#03e50d7d



📊 Support interaction tagged with:
   - Interaction Type: product_recommendation
   - Outcome: purchase_intent
   - Product Category: laptops
   - Customer Sentiment: positive


### Consultas Avançadas de Metadados

Analise padrões de suporte e comportamento do cliente:

In [36]:
from bedrock_agentcore.memory.models import EventMetadataFilter, LeftExpression, RightExpression, OperatorType

try:
    # Query product recommendation interactions
    recommendation_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "interaction_type"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "product_recommendation"}}
            }
        ]
    )
    
    print(f"\n🛍️ Found {len(recommendation_events)} product recommendation interaction(s)")
    
    # Query positive sentiment interactions
    positive_events = customer_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "customer_sentiment"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "positive"}}
            }
        ]
    )
    
    print(f"😊 Found {len(positive_events)} positive sentiment interaction(s)")
    
    print("\n💡 Advanced analytics use cases:")
    print("   - Track conversion rates from recommendations to purchases")
    print("   - Analyze customer sentiment trends over time")
    print("   - Identify most effective support interaction types")
    print("   - Measure support tier performance")
    print("   - Generate detailed customer journey reports")
    print("   - Optimize product recommendation strategies")
    
except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print(f"Note: Metadata filtering requires events with metadata tags")

2026-03-24 19:08:18,635 - INFO - Retrieved total of 1 events



🛍️ Found 1 product recommendation interaction(s)


2026-03-24 19:08:18,821 - INFO - Retrieved total of 1 events


😊 Found 1 positive sentiment interaction(s)

💡 Advanced analytics use cases:
   - Track conversion rates from recommendations to purchases
   - Analyze customer sentiment trends over time
   - Identify most effective support interaction types
   - Measure support tier performance
   - Generate detailed customer journey reports
   - Optimize product recommendation strategies


#### Tutorial de Suporte ao Cliente concluído! 🎉 
Principais conclusões:
- Hooks de memória gerenciam automaticamente o contexto do cliente entre sessões de suporte usando MemorySessionManager
- Estratégias integradas (SemanticStrategy, UserPreferenceStrategy) capturam pedidos, preferências e fatos sem necessidade de IAM roles
- Agentes podem fornecer suporte personalizado com base no histórico do cliente
- Ferramentas podem ser integradas para consulta de pedidos e funcionalidade de busca na web
- O suporte ao cliente se torna mais eficiente com memória persistente
- **Ramificação permite testar abordagens alternativas de suporte e caminhos de escalonamento**
- **Metadados fornecem análises abrangentes de suporte e rastreamento da jornada do cliente**
- **Estratégias integradas simplificam a configuração eliminando a necessidade de configuração de IAM role**

## Limpeza

### Opcional: Excluir Recurso de Memória

In [37]:
# Clean up all resources created during notebook execution
print("=== Limpeza de Recursos ===")

# Step 1: Delete all events from the customer session (main branch)
try:
    events = customer_session.list_events()
    if events:
        logger.info(f"🗑️ Deleting {len(events)} events from main branch...")
        for event in events:
            event_id = event.eventId if hasattr(event, 'eventId') else event.get('eventId', event.get('id'))
            if event_id:
                customer_session.delete_event(event_id)
        logger.info("✅ Main branch events deleted")
    else:
        logger.info("ℹ️ No events found in main branch.")
except Exception as e:
    logger.warning(f"⚠️ Could not delete main branch events: {e}")

# Step 2: Delete events from the premium-support branch (if created)
try:
    branch_events = customer_session.list_events(branch_name="premium-support")
    if branch_events:
        logger.info(f"🗑️ Deleting {len(branch_events)} events from premium-support branch...")
        for event in branch_events:
            event_id = event.eventId if hasattr(event, 'eventId') else event.get('eventId', event.get('id'))
            if event_id:
                customer_session.delete_event(event_id, branch_name="premium-support")
        logger.info("✅ Premium-support branch events deleted")
except Exception as e:
    logger.warning(f"⚠️ Could not delete branch events: {e}")

# Step 3: Delete the Memory resource and wait for completion
# This removes the memory, all strategies, actors, sessions, events, and memory records
try:
    logger.info(f"🗑️ Deleting memory resource: {memory_id}")
    memory_manager.delete_memory_and_wait(memory_id)
    logger.info(f"✅ Memory resource fully deleted: {memory_id}")
except Exception as e:
    logger.error(f"❌ Failed to delete memory: {e}")

print("=== Limpeza concluída ===")

=== Limpeza de Recursos ===


2026-03-24 19:08:19,035 - INFO - Retrieved total of 7 events
2026-03-24 19:08:19,036 - INFO - 🗑️ Deleting 7 events from main branch...
2026-03-24 19:08:19,036 - INFO -   -> Deleting event by ID: 0000001774390098150#03e50d7d...
2026-03-24 19:08:19,244 - INFO -      ✅ Event deleted successfully.
2026-03-24 19:08:19,245 - INFO -   -> Deleting event by ID: 0000001774390097623#e3906d1c...
2026-03-24 19:08:19,443 - INFO -      ✅ Event deleted successfully.
2026-03-24 19:08:19,443 - INFO -   -> Deleting event by ID: 0000001774390097126#316543f4...
2026-03-24 19:08:19,650 - INFO -      ✅ Event deleted successfully.
2026-03-24 19:08:19,651 - INFO -   -> Deleting event by ID: 0000001774390093201#c0640ebe...
2026-03-24 19:08:19,835 - INFO -      ✅ Event deleted successfully.
2026-03-24 19:08:19,836 - INFO -   -> Deleting event by ID: 0000001774390090327#a9bc198c...
2026-03-24 19:08:20,023 - INFO -      ✅ Event deleted successfully.
2026-03-24 19:08:20,023 - INFO -   -> Deleting event by ID: 00000

=== Limpeza concluída ===
